# An-Ra V4 — TPU Training Launch Console (`core-vnext`)

**One path:** this notebook → `training.train_xla` → 8 × v5e workers → schema-v3 exact-resume checkpoints.

Enforced by this console:
- Parent checkpoint is **explicit** (`ANRA_TPU_CHECKPOINT`). No highest-step auto-pick.
- Token pack is **semantically verified** before any worker spawns.
- A run **receipt** binds commit / pack SHA / parent parameter SHA / config.
- Candidates save sparsely and are never overwritten.
- The written recovery artifact is strictly reloaded and contract-checked.


In [ ]:
# 1. TPU runtime preflight — fail closed.
import importlib.metadata, importlib.util, os, sys
os.environ['PJRT_DEVICE'] = 'TPU'
if importlib.util.find_spec('torch_xla') is None:
    raise RuntimeError('TPU not attached. Settings > Accelerator > TPU v5e-8, verify, restart, Run All.')
import torch
print({'pjrt': os.environ['PJRT_DEVICE'], 'torch': torch.__version__,
       'torch_xla': importlib.metadata.version('torch-xla'),
       'note': 'device acquisition is deferred to clean spawned workers'})

In [ ]:
# 2. Clone the exact training branch and record its commit for provenance.
import json, os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
REPO_REF = 'core-vnext'
REPO = Path('/kaggle/working/anra')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', f'origin/{REPO_REF}'], check=True)
sys.path.insert(0, str(REPO))
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
os.environ['ANRA_SOURCE_COMMIT'] = SOURCE_COMMIT
print(json.dumps({'repo': str(REPO), 'commit': SOURCE_COMMIT[:12], 'branch': REPO_REF}))

In [ ]:
# 3. Locate checkpoint + pack. FAIL CLOSED on ambiguity; never guess.
import hashlib, json, os, tarfile
from pathlib import Path
INPUT_ROOT = Path('/kaggle/input')

# REQUIRED: verified schema-3 full-resume checkpoint at global/optimizer step 21,800.
ANRA_TPU_CHECKPOINT = os.environ.get('ANRA_TPU_CHECKPOINT', 'anra-v4-tpu-latest (2).pt')
EXPECTED_CHECKPOINT_SHA256 = 'ae64a43839e344e966c5619bde337943577005d48bf5ec35ed45acfa84af371a'

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def safe_extract(archive, destination):
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with tarfile.open(archive, 'r:gz') as bundle:
        for member in bundle.getmembers():
            target = (destination / member.name).resolve()
            if member.issym() or member.islnk():
                raise RuntimeError(f'archive links are refused: {member.name}')
            if root not in target.parents and target != root:
                raise RuntimeError(f'unsafe archive member: {member.name}')
        bundle.extractall(destination)

def find_checkpoint():
    if not ANRA_TPU_CHECKPOINT:
        raise RuntimeError(
            'Set ANRA_TPU_CHECKPOINT to the EXACT parent file name. '
            'Use the verified step-21800 full-resume checkpoint; never auto-select by highest step.')
    matches = [p for p in INPUT_ROOT.rglob('*.pt') if p.name == ANRA_TPU_CHECKPOINT]
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly 1 match for {ANRA_TPU_CHECKPOINT!r}, found {len(matches)}.')
    return matches[0]

def find_pack():
    campaign_manifests = sorted(INPUT_ROOT.rglob('pack_manifest.json'))
    if len(campaign_manifests) == 1:
        train_root = campaign_manifests[0].parent / 'train'
        if not (train_root / 'manifest.json').is_file():
            raise RuntimeError('Campaign pack lacks train/manifest.json')
        return train_root
    if len(campaign_manifests) > 1:
        raise RuntimeError(f'Expected exactly 1 campaign pack, found {len(campaign_manifests)}.')
    archives = sorted(INPUT_ROOT.rglob('*.tar.gz'))
    if len(archives) != 1:
        raise RuntimeError(f'Expected exactly 1 pack archive (*.tar.gz), found {len(archives)}. Plain text datasets are refused by design.')
    dest = Path('/kaggle/working/pack')
    safe_extract(archives[0], dest)
    if not (dest / 'manifest.json').is_file():
        raise RuntimeError('Archive lacks manifest.json - build it with training.pack_verify.build_manifest.')
    return dest

CHECKPOINT = find_checkpoint()
PACK_ROOT = find_pack()
CHECKPOINT_SHA = sha256_file(CHECKPOINT)
if CHECKPOINT_SHA != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(f'Checkpoint SHA-256 mismatch: {CHECKPOINT_SHA}')
print(json.dumps({'checkpoint': str(CHECKPOINT), 'checkpoint_sha256': CHECKPOINT_SHA[:16],
                  'pack_root': str(PACK_ROOT)}, indent=2))


In [ ]:
# 4. Run configuration - one place, printed for visual confirmation.
import json
CONFIG = {
    'max_steps': 0,               # 0 = every complete unique-data update once
    'max_minutes': 430,           # leave headroom inside the Kaggle session
    'batch_size': 1,              # per core
    'grad_accum_steps': 8,
    'learning_rate': 2e-4,
    'weight_decay': 0.1,
    'save_interval': 200,         # recovery checkpoint cadence
    'candidate_interval': 1000,   # immutable full-resume recovery candidates
    'gradient_checkpointing': False, # avoid pathological XLA reentrant compile
    'warmup_fraction': 0.0,       # never re-warm an already-trained model
    'decay_fraction': 0.1,        # WSD decay lands at the pack boundary
    'log_interval': 1,
    'seed': 1301,
}
print(json.dumps(CONFIG, indent=2))


In [ ]:
# 5. Verify the pack FAIL-CLOSED before any GPU-hour is spent.
from pathlib import Path
from training.pack_verify import PackVerificationError, verify_pack
try:
    pack = verify_pack(Path(PACK_ROOT))
except PackVerificationError as exc:
    raise SystemExit(f'REFUSING TO TRAIN: {exc}') from exc
available_pack_steps = pack.total_windows // (8 * CONFIG['batch_size'] * CONFIG['grad_accum_steps'])
if CONFIG['max_steps'] and CONFIG['max_steps'] > available_pack_steps:
    raise RuntimeError('max_steps exceeds complete unique-data updates')
CONFIG['_pack_total_steps'] = CONFIG['max_steps'] or available_pack_steps
print(f'PACK VERIFIED: {len(pack.shard_paths)} shards | '
      f'{pack.total_tokens:,} tokens | ~{pack.total_windows:,} unique windows')

In [ ]:
# 6. PREFLIGHT (CPU): pack semantics + parent restore through the canonical
# path BEFORE any worker spawns. Writes the run receipt. Fails Run All on any error.
import json
from pathlib import Path
from training.train_xla import preflight, write_run_receipt
from anra_core.config import CANONICAL_CONFIG

identity = preflight(
    dataset_path=PACK_ROOT,
    checkpoint_path=CHECKPOINT,
    block_size=CANONICAL_CONFIG.block_size,
    vocab_size=CANONICAL_CONFIG.vocab_size,
    expected_resume_step=21_800,
    start_new_pack=False,
    allow_legacy_resume=False,
)
print(json.dumps(identity, indent=2))
print()
print('VISUAL CHECK - is this the parent you intended?')
print(f"  parent step: {identity['parent_global_step']}")
print(f"  param sha:   {identity['parent_parameter_sha256'][:16]}")
print(f"  pack sha:    {identity['pack_manifest_sha256'][:16]}")
print(f"  windows:     {identity['pack_windows']:,}")

assert identity['parent_global_step'] == 21_800
assert identity['parent_parameter_sha256'] == '6d94f1f6a085643d37b2f92b37107ebf513b68c3209d64e60ae07b55f117d2b4'
EXPECTED_FINAL_STEP = 22_517
RUN_DIR = Path('/kaggle/working/runs/run-006')
if RUN_DIR.exists():
    raise RuntimeError(f'{RUN_DIR} already exists (duplicate Run All?). Delete it or use run-002.')
receipt = write_run_receipt(RUN_DIR, identity_block=identity, config=CONFIG, world_size=8)
print(f'receipt: {receipt}')


In [ ]:
# 7. Launch training (8 workers via torch_xla.launch inside train_xla).
# Recovery checkpoint every save_interval; immutable candidates every candidate_interval.
import subprocess, sys
command = [sys.executable, '-m', 'training.train_xla',
    '--dataset-path', str(PACK_ROOT),
    '--output-checkpoint', str(RUN_DIR / 'anra-v4-tpu-latest.pt'),
    '--resume-from', str(CHECKPOINT),
    '--expected-resume-step', '21800',
    '--no-gradient-checkpointing',
    '--max-steps', str(CONFIG['max_steps']),
    '--max-minutes', str(CONFIG['max_minutes']),
    '--batch-size', str(CONFIG['batch_size']),
    '--grad-accum-steps', str(CONFIG['grad_accum_steps']),
    '--learning-rate', str(CONFIG['learning_rate']),
    '--weight-decay', str(CONFIG['weight_decay']),
    '--warmup-fraction', str(CONFIG['warmup_fraction']),
    '--decay-fraction', str(CONFIG['decay_fraction']),
    '--save-interval', str(CONFIG['save_interval']),
    '--candidate-interval', str(CONFIG['candidate_interval']),
    '--log-interval', str(CONFIG['log_interval']),
    '--seed', str(CONFIG['seed'])]
print(' '.join(command))
result = subprocess.run(command, cwd=REPO)
if result.returncode != 0:
    raise RuntimeError(f'training exited {result.returncode} - inspect log above')

# Verify before declaring the artifact downloadable. This catches the exact
# stale-parent/optimizer-step failure that invalidated earlier runs.
FINAL_CHECKPOINT = RUN_DIR / 'anra-v4-tpu-latest.pt'
DOWNLOAD_MANIFEST = RUN_DIR / 'download-manifest.json'
verify_command = [sys.executable, str(REPO / 'scripts' / 'verify_checkpoint_for_download.py'),
    '--checkpoint', str(FINAL_CHECKPOINT), '--parent', str(CHECKPOINT),
    '--expected-step', str(EXPECTED_FINAL_STEP),
    '--manifest', str(DOWNLOAD_MANIFEST)]
subprocess.run(verify_command, cwd=REPO, check=True)

# Keep the kernel busy after verification so Kaggle cannot erase /kaggle/working
# during its short idle window. Download from Output > runs > run-006, then
# interrupt this cell only after the local file has been verified.
import time
from IPython.display import FileLink, display
display(FileLink(FINAL_CHECKPOINT))
print('DOWNLOAD READY: verified checkpoint is protected while this cell runs.', flush=True)
print('DO NOT stop the session until the local SHA-256 matches download-manifest.json.', flush=True)
while True:
    print(f'[checkpoint keepalive] {time.strftime("%Y-%m-%d %H:%M:%S")} '          f'{FINAL_CHECKPOINT} size={FINAL_CHECKPOINT.stat().st_size:,}', flush=True)
    time.sleep(60)


In [ ]:
# 8. Verify artifacts AND prove this run changed the parent weights.
import hashlib, json
from pathlib import Path
from anra_core.checkpoint import load_core_checkpoint
latest = RUN_DIR / 'anra-v4-tpu-latest.pt'
parent_model, parent_payload, parent_identity = load_core_checkpoint(CHECKPOINT)
trained_model, payload, trained_identity = load_core_checkpoint(latest)
assert trained_identity.artifact_class == 'full_resume'
assert payload.get('checkpoint_schema_version') == 3
assert payload.get('global_step') == EXPECTED_FINAL_STEP
assert payload.get('optimizer_state_dict', {}).get('state')
assert payload.get('trainer_state', {}).get('pack_manifest_sha256') == pack.manifest_sha256
assert payload.get('lr_schedule', {}).get('name') == 'wsd_pack_v1'
assert trained_identity.parameter_sha256 != parent_identity.parameter_sha256, (
    'INVALID RUN: final model parameters are byte-identical to the parent')
optimizer_steps = [int(state['step']) for state in payload['optimizer_state_dict']['state'].values()]
assert optimizer_steps and max(optimizer_steps) == payload['global_step'], (
    f"INVALID RUN: Adam state stopped at {max(optimizer_steps, default=0)}, "
    f"checkpoint claims {payload['global_step']}")
del parent_model, trained_model
print(json.dumps({
    'latest_step': payload.get('global_step'),
    'weights_changed_from_parent': True,
    'parent_parameter_sha256': parent_identity.parameter_sha256[:16],
    'trained_parameter_sha256': trained_identity.parameter_sha256[:16],
    'optimizer_state_step': max(optimizer_steps),
    'latest_sha256': hashlib.sha256(latest.read_bytes()).hexdigest()[:16],
    'candidates': sorted(p.name for p in (RUN_DIR / 'candidates').glob('*.pt')), 
}, indent=2))
